In [3]:
import os
import sys
from pathlib import Path

# 必须在 CUDA 初始化前设置
os.environ["CUDA_VISIBLE_DEVICES"] = (
    "GPU-0249a4e6-0715-37a5-b8f9-6cf8cfc674d1"
)

repo = Path("/home/v-jinyuning/LLMs-from-scratch")
sys.path.insert(0, str(repo / "ch05/01_main-chapter-code"))

import torch
import tiktoken
from previous_chapters import GPTModel, generate_text_simple

assert torch.cuda.is_available(), "当前内核无法使用 CUDA"
device = torch.device("cuda:0")

print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0))

Python: /home/v-jinyuning/miniconda3/envs/llmscratch/bin/python
GPU: NVIDIA GeForce RTX 5090


In [4]:
config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

weight_path = (
    repo
    / "ch05/02_alternative_weight_loading/gpt2-small-124M.pth"
)

model = GPTModel(config)

weights = torch.load(
    weight_path,
    map_location="cpu",
    weights_only=True,
)
result = model.load_state_dict(weights)
del weights

model = model.to(device)
model.eval()

print(result)
print("模型所在设备:", next(model.parameters()).device)

<All keys matched successfully>
模型所在设备: cuda:0


In [5]:
import os
import hashlib
from pathlib import Path

source_dir = Path(
    "/home/v-jinyuning/LLMs-from-scratch/"
    "ch05/02_alternative_weight_loading/tokenizer_files"
)
cache_dir = source_dir / "cache"

# URL 和校验值来自 tiktoken 官方 GPT-2 配置
base_url = "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/"
expected_hashes = {
    "vocab.bpe":
        "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
    "encoder.json":
        "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783",
}

cache_dir.mkdir(parents=True, exist_ok=True)

for filename, expected_hash in expected_hashes.items():
    data = (source_dir / filename).read_bytes()

    # 防止误存成网页或上传不完整
    actual_hash = hashlib.sha256(data).hexdigest()
    assert actual_hash == expected_hash, f"{filename} 校验失败，请检查下载文件"

    # tiktoken 使用下载 URL 的 SHA-1 作为缓存文件名
    cache_name = hashlib.sha1((base_url + filename).encode()).hexdigest()
    (cache_dir / cache_name).write_bytes(data)
    print(filename, "校验通过，缓存已准备")

os.environ["TIKTOKEN_CACHE_DIR"] = str(cache_dir)

vocab.bpe 校验通过，缓存已准备
encoder.json 校验通过，缓存已准备


In [6]:
tokenizer = tiktoken.get_encoding("gpt2")

In [7]:
prompt = "Every effort moves you"
token_ids = tokenizer.encode(prompt)

inputs = torch.tensor(
    token_ids,
    dtype=torch.long,
    device=device,
).unsqueeze(0)

print("提示词:", prompt)
print("Token IDs:", token_ids)
print("输入 shape:", inputs.shape)

提示词: Every effort moves you
Token IDs: [6109, 3626, 6100, 345]
输入 shape: torch.Size([1, 4])


In [8]:
with torch.inference_mode():
    logits = model(inputs)
    print("Logits shape:", logits.shape)  # [1, T, 50257]

    output_ids = generate_text_simple(
        model=model,
        idx=inputs,
        max_new_tokens=30,
        context_size=config["context_length"],
    )

output_text = tokenizer.decode(output_ids[0].tolist())
print("\n生成结果：\n", output_text)

Logits shape: torch.Size([1, 4, 50257])

生成结果：
 Every effort moves you forward.

The first step is to understand the importance of your work.

The second step is to understand the importance of your work.
